# Language Models

In [1]:
import dspy
llama32 = dspy.LM('ollama_chat/llama3.2', api_base='http://localhost:11434', api_key='')
gpt_oss = dspy.LM('ollama_chat/gpt-oss:latest', api_base='http://localhost:11434', api_key='')

## Calling the LM directly

In [2]:
with dspy.context(lm=llama32):
    llama32("Say this is a test!", temperature=0.7)  # => ['This is a test!']
    output = llama32(messages=[{"role": "user", "content": "Say this is a test!"}])  # => ['This is a test!']
    print(output)

["It looks like we're starting fresh. How can I assist you today?"]


## Using the LM with DSPy modules

In [3]:
with dspy.context(lm=llama32):

    # Define a module (ChainOfThought) and assign it a signature (return an answer, given a question).
    qa = dspy.ChainOfThought('question -> answer')

    # Run with the default LM configured with `dspy.context` above.
    response = qa(question="How many floors are in the castle David Gregory inherited?")
    print(response.answer)

The Banns Castle has 4 floors.


## Using multiple LMs

In [4]:
with dspy.context(lm=llama32):
    response = qa(question="How many floors are in the castle David Gregory inherited?")
    print('llama3.2:', response.answer)

with dspy.context(lm=gpt_oss):
    response = qa(question="How many floors are in the castle David Gregory inherited?")
    print('gpt-oss:', response.answer)

llama3.2: The Banns Castle has 4 floors.
gpt-oss: 5


## Configuring LM generation

In [5]:
llama32_cache_disabled = dspy.LM('ollama_chat/llama3.2', api_base='http://localhost:11434', api_key='', temperature=0.9, max_tokens=3000, stop=None, cache=False)

In [7]:
with dspy.context(lm=llama32):
    output = llama32("Say this is a test!", rollout_id=1, temperature=1.0)
    print(output)

["It looks like you're testing to see if I'm working properly. Well, I'm happy to report that I am! How can I assist you today?"]


In [8]:
with dspy.context(lm=llama32):
    predict = dspy.Predict("question -> answer", rollout_id=1, temperature=1.0)

In [ ]:
with dspy.context(lm=llama32):
    predict = dspy.Predict("question -> answer")
    output = predict(question="What is 1 + 52?", config={"rollout_id": 5, "temperature": 1.0})
    print(output)

Prediction(
    answer='The sum of 1 and 52 is 53.'
)


## Inspecting output and usage metadata

In [10]:
with dspy.context(lm=llama32):
    print(len(llama32.history))  # e.g., 3 calls to the LM

    print(llama32.history[-1].keys())  # access the last call to the LM, with all metadata

7
dict_keys(['prompt', 'messages', 'kwargs', 'response', 'outputs', 'usage', 'cost', 'timestamp', 'uuid', 'model', 'response_model', 'model_type'])


## Using the Responses API

In [11]:
import dspy

# Configure DSPy to use the Responses API for your language model
dspy.configure(
    lm=dspy.LM(
        "ollama_chat/llama3.2",
        model_type="responses",
        temperature=1.0,
        max_tokens=16000,
    ),
)